In [2]:

import os
import sys
print("Check that we are running in the venv")
print(f'environment: {sys.executable}')

Check that we are running in the venv
environment: /Users/idekeradmin/Dropbox/GitHub/beng_203_project/venv/bin/python


In [3]:
# import utilities from the correct relative path
import silver_seq_utils as utils
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

# load silver seq data
X, silver_seq_counts, silver_seq_meta = utils.load_silver_seq_data()
mappings = pd.read_csv("data/" + 'gene_mappings.csv')
gene_sets = {}
mappings.head()

,query,_id,_score,name,symbol,entrezgene,notfound
0,ENSG00000223972,ENSG00000223972,10.614574,DEAD/H-box helicase 11 like 1 (pseudogene),DDX11L1,NaN,NaN
1,ENSG00000227232,ENSG00000227232,10.614576,"WASP family homolog 7, pseudogene",WASH7P,NaN,NaN
2,ENSG00000278267,102466751,32.905167,microRNA 6859-1,MIR6859-1,102466751.0,NaN
3,ENSG00000243485,ENSG00000243485,32.905178,MIR1302-2 host gene,MIR1302-2HG,NaN,NaN
4,ENSG00000274890,NaN,NaN,NaN,NaN,NaN,True


# Gene Sets From NDEx Networks

NDEx, The Network Data Exchange, is a public commons for storing, sharing, publishing, and using networks. It hosts multiple public collections of pathways and interaction networks. We derived gene sets from these networks by:
- pathways related to AD, neuron cell death, and RNA
- mitochondrial genes involved in oxidation and apoptosis, potentially released as part of mitochondria going to the bloodstream in neuron lysis.
- subnetworks derived from queries for interactions of "seed" genes in PPI networks (Using BioPlex, based on experimental data, avoiding curation bias).

Specifically:
- interaction neighborhoods in bioplex around:
  - AD gwas genes
  - DEGs in the study
  - PHGDH
- Pathway networks

    - autophagy
    - neurodegeneration
    - alzheimer's
    - inflammation types
    - ox-phos
    - mtor
    - cell cycle
    - hypoxia
    - metabolic
    - cell death
        - caspase
    - rna binding
    - cytosolic rna sensing
    - growth and proliferation



In [4]:
networks = {
     "49e43d68-939b-11ea-aaef-0ac135e8bacf": {
         "name": "AD network",
         "node_name_field": "livia name",
         "description": "AD network"
     },
     "2098de1c-a927-11eb-9e72-0ac135e8bacf": {
         "name": "neurodegen-autophagy",
         "node_name_field": "name",
         "description": "neurodegeneration genes adjacent to autophagy ppi"
     },
     "eedec451-d9b7-11e9-bb65-0ac135e8bacf":{
         "name": "oxidative demethylation",
          "node_name_field": "name",
         "description": "GO:0070989 (oxidative demethylation)"
     },
     "09f15eff-84ab-11ec-b3be-0ac135e8bacf":{
         "name": "oxidative phosphorylation",
          "node_name_field": "name",
         "description": "GO:0006119 (oxidative phosphorylation)"
     },
     "e5f038e8-bf17-11ea-aaef-0ac135e8bacf":{
         "name": "oxidative RNA demethylation",
          "node_name_field": "name",
         "description": "GO:0035513 (oxidative RNA demethylation)"
     },
     "d49b187c-9ba4-11ee-8a13-005056ae23aa":{
         "name": "AKT/MTOR pathway I",
          "node_name_field": "name",
         "description": "AKT/MTOR pathway I"
     },
     "ac88cce0-7dcf-11ea-aaef-0ac135e8bacf":{
         "name": "release of cytochrome c from mitochondria",
          "node_name_field": "name",
         "description": "GO:0090200 (positive regulation of release of cytochrome c from mitochondria)"
     },
     "a9ffc83d-36dc-11ec-b3be-0ac135e8bacf":{
         "name": "autophagic cell death",
          "node_name_field": "name",
         "description": "GO:0048102 (autophagic cell death)"
     }

}

def get_node_names(cx2_network, name_field="name"):
    node_names = []
    for id, node in cx2_network.get_nodes().items():
        properties = node['v']
        name = properties[name_field]
        node_names.append(name)
    return node_names

def get_ndex_gene_set(network_id, name, node_name_field="name", description=None):
  cx2_network = utils.get_ndex_network_by_id(network_id)
  symbol_list = get_node_names(cx2_network, node_name_field)
  id_list = mappings.loc[mappings["symbol"].isin(symbol_list), "query"].to_list()
  return {
      "id_list": " ".join(id_list),
      "gene_list": " ".join(symbol_list),
      "name": name,
      "description": description
  }

for network_id, network_info in networks.items():
  gene_sets[network_id] = get_ndex_gene_set(network_id, **network_info)

for network_id, set in gene_sets.items():
  print(f'{network_id} : {set}')

49e43d68-939b-11ea-aaef-0ac135e8bacf : {'id_list': 'ENSG00000241563 ENSG00000181773 ENSG00000116133 ENSG00000134247 ENSG00000117362 ENSG00000185499 ENSG00000198400 ENSG00000162736 ENSG00000152061 ENSG00000133069 ENSG00000092969 ENSG00000143801 ENSG00000138095 ENSG00000121966 ENSG00000138449 ENSG00000172020 ENSG00000082701 ENSG00000163902 ENSG00000157005 ENSG00000197386 ENSG00000163956 ENSG00000168421 ENSG00000163697 ENSG00000145283 ENSG00000145335 ENSG00000182168 ENSG00000168785 ENSG00000171497 ENSG00000171368 ENSG00000142319 ENSG00000064692 ENSG00000113558 ENSG00000113108 ENSG00000183775 ENSG00000156475 ENSG00000113758 ENSG00000124788 ENSG00000137312 ENSG00000204388 ENSG00000095970 ENSG00000197442 ENSG00000130396 ENSG00000106536 ENSG00000106089 ENSG00000106333 ENSG00000105835 ENSG00000158457 ENSG00000008056 ENSG00000078596 ENSG00000123560 ENSG00000168484 ENSG00000277586 ENSG00000120885 ENSG00000147955 ENSG00000107282 ENSG00000136854 ENSG00000177697 ENSG00000110651 ENSG00000166313 ENSG

# Other Gene Sets

We derived gene sets from selected literature and AD genes identified in GWAS studies.

Mitochondiral genes, some of the most variable genes in the data

------------
  Hu R, Yu Q, Zhou S, Yin Y, Hu R, Lu H and Hu B (2020) Co-expression Network Analysis Reveals Novel Genes Underlying Alzheimer’s Disease Pathogenesis. Front. Aging Neurosci. 12:605961. doi: 10.3389/fnagi.2020.605961

  NO2, ELAVL4, SNAP91, NEFM

----

Zebardast, F., Riethmüller, M.P.S. & Nowick, K. Integrative gene co-expression network analysis reveals protein-coding and LncRNA genes associated with Alzheimer’s disease pathology. Sci Rep 15, 43395 (2025). https://doi.org/10.1038/s41598-025-30392-9

Clusters in excel spreadsheet, selecting:
-

In [5]:
def add_gene_set(gene_sets, set_id, id_list, name="", gene_list="", description=""):
  gene_sets[set_id] = {
      "id_list": id_list,
      "gene_list": gene_list,
      "name": name,
      "description": description
  }

def add_ids(gene_set_dict, mappings):
  if gene_set_dict.get("id_list") is None:
    gene_set_dict["id_list"] = " ".join(mappings.loc[mappings["symbol"].isin(gene_set_dict["gene_list"]), "query"].to_list())
    gene_set_dict["gene_list"] = " ".join(gene_set_dict["gene_list"])
  return gene_set_dict

In [6]:


mt_gene_ids = [
    "ENSG00000210049", # MT-TF
    "ENSG00000211459", # MT-RNR1
    "ENSG00000210077", # MT-TV
    "ENSG00000210082", # MT-RNR2
    "ENSG00000209082", # MT-TL1
    "ENSG00000198888", # MT-ND1
    "ENSG00000210100", # MT-TI
    "ENSG00000210107", # MT-TQ
    "ENSG00000210112", # MT-TM
    "ENSG00000198763", # MT-ND2
    "ENSG00000210117", # MT-TW
    "ENSG00000210127", # MT-TA
    "ENSG00000210135", # MT-TN
    "ENSG00000210140", # MT-TC
    "ENSG00000210144", # MT-TY
    "ENSG00000198804", # MT-CO1
    "ENSG00000210151", # MT-TS1
    "ENSG00000210154", # MT-TD
    "ENSG00000198712", # MT-CO2
    "ENSG00000210156", # MT-TK
    "ENSG00000228253", # MT-ATP8
    "ENSG00000198899", # MT-ATP6
    "ENSG00000198938", # MT-CO3
    "ENSG00000210164", # MT-TG
    "ENSG00000198840", # MT-ND3
    "ENSG00000210174", # MT-TR
    "ENSG00000212907", # MT-ND4L
    "ENSG00000198886", # MT-ND4
    "ENSG00000210176", # MT-TH
    "ENSG00000210184", # MT-TS2
    "ENSG00000210191", # MT-TL2
    "ENSG00000198786", # MT-ND5
    "ENSG00000198695", # MT-ND6
    "ENSG00000210194", # MT-TE
    "ENSG00000198727", # MT-CYB
    "ENSG00000210195", # MT-TT
    "ENSG00000210196"  # MT-TP
    ]

add_gene_set(gene_sets, "MT Genes", " ".join(mt_gene_ids), name="MT Genes", gene_list="None", description="Mitochondrial Genes")

In [7]:
phgdh_interactors = ["WDTC1", "PHGDH", "ARIH1", "EPN1", "EPN3", "GALNT11", "MCU" "MYADM"  "S100A6" "SLC31A1"]
phgdh_interactor_ids = mappings.loc[mappings["symbol"].isin(phgdh_interactors), "query"].to_list()


add_gene_set(gene_sets, "PHGDH interactors", " ".join(phgdh_interactor_ids), name="PHGDH interactors", gene_list=" ".join(phgdh_interactors), description="Binds PHGDH in BioPlex")
add_gene_set(gene_sets, "PHGDH", 'ENSG00000092621', name="PHGDH", gene_list="PHGDH", description="Just PHGDH")

In [8]:
gene_sets["s13_ad_associated_lncrnas"] = {
    "id_list": [
        "ENSG00000278768",
        "ENSG00000246790",
        "ENSG00000236824",
        "ENSG00000240498",
        "ENSG00000248587",
        "ENSG00000225978",
        "ENSG00000231133",
        "ENSG00000242125",
        "ENSG00000242808",
        "ENSG00000281721",
        "ENSG00000261340",
        "ENSG00000226029",
        "ENSG00000259125",
        "ENSG00000225783",
        "ENSG00000245532",
        "ENSG00000260924",
        "ENSG00000241560",
        "ENSG00000264745",
        "ENSG00000152931",
        "ENSG00000224078",
        "ENSG00000234741",
        "ENSG00000236901",
        "ENSG00000240562",
        "ENSG00000229807",
        "ENSG00000277209",
        "ENSG00000214548",
        "ENSG00000245573",
        "ENSG00000251562",
        "ENSG00000236824",
        "ENSG00000287997",
        "ENSG00000259125",
        "ENSG00000248587",
        "ENSG00000255717",
        "ENSG00000245573",
        "ENSG00000183242",
        "ENSG00000237737",
        "ENSG00000280809",
        "ENSG00000261610",
        "ENSG00000261087",
        "ENSG00000259976",
        "ENSG00000271147",
        "ENSG00000259976",
    ],
    "gene_list": [
        "BACE1-AS",
        "SORL1-AS1",
        "BCYRN1",
        "CDKN2B-AS1",
        "GDNFOS",
        "HAR1A",
        "HAR1B",
        "SNHG3",
        "SOX2-OT",
        "LINC01080",
        "LINC01616",
        "LINC01772",
        "LRP1-AS",
        "MIAT",
        "NEAT1",
        "LINC01311",
        "ZBTB20-AS1",
        "TTC39C-AS1",
        "PART1",
        "SNHG14",
        "GAS5",
        "MIR600HG",
        "RP11-59J16.2",
        "XIST",
        "Rpph1",
        "MEG3",
        "BDNF-AS",
        "MALAT1",
        "BC200",
        "EBF3-AS1",
        "LRP1-AS",
        "GDNF-AS1",
        "SNHG1",
        "BDNF-AS",
        "WT1-AS",
        "DCTN1-AS1",
        "LINC00836",
        "Lnc-MIS18A-1",
        "Lnc-ZNF706-1",
        "NNT-AS1",
        "ARMCX5-GPRASP2",
        "Lnc-ZBTB20-1",
    ],
    "name": "S13 AD-associated lncRNAs",
    "description": "AD-associated lncRNAs from Supplementary Table S13, with entries lacking Ensembl IDs removed. Table-order duplicates are retained."
}

gene_set_dict = gene_sets["s13_ad_associated_lncrnas"]
gene_set_dict["id_list"] = " ".join(gene_set_dict["id_list"])
gene_set_dict["gene_list"] = " ".join(gene_set_dict["gene_list"])

Zebardast, F., Riethmüller, M.P.S. & Nowick, K. Integrative gene co-expression network analysis reveals protein-coding and LncRNA genes associated with Alzheimer’s disease pathology. Sci Rep 15, 43395 (2025). https://doi.org/10.1038/s41598-025-30392-9

In [9]:
gene_sets["text_ad_only_functional_lncrnas"] = {
    "gene_list": [
        "FBXW7-AS1",
        "Lnc-PDE4D-2",
        "Lnc-KLHL11-1",
        "Lnc-NIF3L1-5",
        "Lnc-KALRN-1",
        "Lnc-CCDC68-1",
        "HSALNG0099005",
        "HSALNG0105911",
        "ZRANB2-DT",
        "ENSG00000287022",
        "SLC8A1-AS1",
    ],
    "name": "AD-only functional lncRNAs",
    "description": "lncRNAs highlighted in the text as functionally assigned exclusively within the AD network, suggesting gain of function in AD.",
    "aggregate": "F",
}
add_ids(gene_sets["text_ad_only_functional_lncrnas"], mappings)

gene_sets["text_downregulated_core_lncrnas"] = {
    "gene_list": [
        "ITFG1-AS1",
        "Lnc-MYCN-6",
        "Lnc-EFR3A-6",
        "HSALNG0089044",
        "Lnc-CCDC68-1",
    ],
    "name": "Downregulated core lncRNAs in AD",
    "description": "Among the highlighted candidate/core lncRNAs, these five were described in the text as dysregulated or downregulated in AD.",
    "aggregate": "F",
}
add_ids(gene_sets["text_downregulated_core_lncrnas"], mappings)

gene_sets["text_ad_only_functional_protein_genes"] = {
    "gene_list": [
        "CHL1",
        "USP33",
        "GRIA2",
        "YARS2",
    ],
    "name": "AD-only functional protein-coding genes",
    "description": "Four protein-coding genes highlighted in the text as annotated to biological processes only within the AD network.",
    "aggregate": "F",
}
add_ids(gene_sets["text_ad_only_functional_protein_genes"], mappings)

gene_sets["text_known_ad_hub_genes"] = {
    "gene_list": [
        "ATL1",
        "AP3M2",
        "ATP6AP2",
        "ATP6V1A",
        "ATP6V1D",
        "ACSL3",
        "CCNB1",
        "GHITM",
        "GRIA2",
        "MDH1",
        "MRPS30",
        "MRPS35",
        "NAPB",
        "NDUFB5",
        "PFN2",
    ],
    "name": "Previously reported AD hub genes",
    "description": "Genes highlighted in the discussion as previously reported hub genes in AD.",
    "aggregate": "F",
}
add_ids(gene_sets["text_known_ad_hub_genes"], mappings)

gene_sets["text_v_atpase_key_genes"] = {
    "gene_list": [
        "ATP6V1B2",
        "ATP6V1A",
        "ATP6AP2",
        "ATP6V1D",
    ],
    "name": "V-ATPase key genes",
    "description": "Four V-ATPase-related genes highlighted in the text as appearing in the study's key gene list.",
    "aggregate": "F",
}
add_ids(gene_sets["text_v_atpase_key_genes"], mappings)

gene_sets["text_therapeutic_target_candidates"] = {
    "gene_list": [
        "ABL1",
        "ACSL3",
        "ATP6V1A",
        "CISD",
        "COQ10B",
    ],
    "name": "Therapeutic target candidates",
    "description": "Genes highlighted in the discussion as promising therapeutic targets for AD.",
    "aggregate": "F",
}
add_ids(gene_sets["text_therapeutic_target_candidates"], mappings)

{'gene_list': 'ABL1 ACSL3 ATP6V1A CISD COQ10B',
 'name': 'Therapeutic target candidates',
 'description': 'Genes highlighted in the discussion as promising therapeutic targets for AD.',
 'aggregate': 'F',
 'id_list': 'ENSG00000115520 ENSG00000123983 ENSG00000114573 ENSG00000097007'}

In [10]:
gene_sets["ad_gwas_harold_2009"] = {
    "gene_list": [
        "APOE",
        "CLU",
        "PICALM",
    ],
    "name": "Harold 2009 AD GWAS loci",
    "description": "Harold et al. reported genome-wide significant association with APOE, CLU, and PICALM.",
    "aggregate": "F"
}
add_ids(gene_sets["ad_gwas_harold_2009"], mappings)

gene_sets["ad_gwas_corneveaux_2010_top_genes"] = {
    "gene_list": [
        "APOE",
        "CLU",
        "PICALM",
        "CR1",
        "CST3",
        "ACE",
    ],
    "name": "Corneveaux 2010 top AD genes",
    "description": "Corneveaux et al. explicitly say they confirmed APOE, CLU, PICALM, CR1, and previously implicated CST3 and ACE.",
    "aggregate": "F"
}
add_ids(gene_sets["ad_gwas_corneveaux_2010_top_genes"], mappings)

gene_sets["ad_gwas_second_wave_review"] = {
    "gene_list": [
        "CD33",
        "MS4A4A",
        "MS4A4E",
        "MS4A6E",
        "ABCA7",
        "CD2AP",
        "EPHA1",
    ],
    "name": "Second-wave AD GWAS genes",
    "description": "Tosto, Reitz et al. review: the second set of large AD GWAS identified CD33, the MS4A4A/MS4A4E/MS4A6E cluster, ABCA7, CD2AP, and EPHA1.",
    "aggregate": "F"
}
add_ids(gene_sets["ad_gwas_second_wave_review"], mappings)

gene_sets["ad_gwas_strongest_after_apoe"] = {
    "gene_list": [
        "BIN1",
        "CLU",
        "CR1",
        "PICALM",
        "MS4A6A",
        "ABCA7",
        "EPHA1",
        "CD33",
        "CD2AP",
    ],
    "name": "Strongest AD GWAS loci after APOE",
    "description": "Vázquez-Higuera et al. explicitly analyzed the 9 AD GWAS loci with the strongest effect sizes after APOE: BIN1, CLU, CR1, PICALM, MS4A6A, ABCA7, EPHA1, CD33, and CD2AP.",
    "aggregate": "F"
}
add_ids(gene_sets["ad_gwas_strongest_after_apoe"], mappings)

gene_sets["ad_microglial_innate_immunity"] = {
    "gene_list": [
        "TREM2",
        "PLCG2",
        "ABI3",
        "INPP5D",
        "SPI1",
        "CD33",
    ],
    "name": "AD microglial innate immunity genes",
    "description": "Sims et al. highlighted TREM2, PLCG2, and ABI3, and also identified INPP5D, SPI1, and CD33 as common-variant AD risk loci in the same microglial-innate-immunity context.",
    "aggregate": "F"
}
add_ids(gene_sets["ad_microglial_innate_immunity"], mappings)



{'gene_list': 'TREM2 PLCG2 ABI3 INPP5D SPI1 CD33',
 'name': 'AD microglial innate immunity genes',
 'description': 'Sims et al. highlighted TREM2, PLCG2, and ABI3, and also identified INPP5D, SPI1, and CD33 as common-variant AD risk loci in the same microglial-innate-immunity context.',
 'aggregate': 'F',
 'id_list': 'ENSG00000168918 ENSG00000095970 ENSG00000066336 ENSG00000197943 ENSG00000108798 ENSG00000105383'}

In [11]:
import json
json.dumps(gene_sets, indent=4)

'{\n    "49e43d68-939b-11ea-aaef-0ac135e8bacf": {\n        "id_list": "ENSG00000241563 ENSG00000181773 ENSG00000116133 ENSG00000134247 ENSG00000117362 ENSG00000185499 ENSG00000198400 ENSG00000162736 ENSG00000152061 ENSG00000133069 ENSG00000092969 ENSG00000143801 ENSG00000138095 ENSG00000121966 ENSG00000138449 ENSG00000172020 ENSG00000082701 ENSG00000163902 ENSG00000157005 ENSG00000197386 ENSG00000163956 ENSG00000168421 ENSG00000163697 ENSG00000145283 ENSG00000145335 ENSG00000182168 ENSG00000168785 ENSG00000171497 ENSG00000171368 ENSG00000142319 ENSG00000064692 ENSG00000113558 ENSG00000113108 ENSG00000183775 ENSG00000156475 ENSG00000113758 ENSG00000124788 ENSG00000137312 ENSG00000204388 ENSG00000095970 ENSG00000197442 ENSG00000130396 ENSG00000106536 ENSG00000106089 ENSG00000106333 ENSG00000105835 ENSG00000158457 ENSG00000008056 ENSG00000078596 ENSG00000123560 ENSG00000168484 ENSG00000277586 ENSG00000120885 ENSG00000147955 ENSG00000107282 ENSG00000136854 ENSG00000177697 ENSG00000110651 E

# Save Gene Sets

In [ ]:

gene_sets_df = pd.DataFrame(gene_sets).T
gene_sets_df.to_csv('gene_sets.csv') 
# this writes to top level to avoid overwriting the 
# file generated at the time we wrote the report
# which is in the 'data' subdirectory
